In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path(r"E:\Projet_Alzheimer")
TABLES = ROOT / "data" / "02_Tables_originales"

if not TABLES.is_dir():
    raise FileNotFoundError(f"Dossier introuvable : {TABLES}")

# Recherche aussi dans les sous-dossiers.
fichiers = sorted(
    p for p in TABLES.rglob("*")
    if p.is_file() and p.suffix.lower() == ".csv"
)

inventaire = pd.DataFrame(
    [
        {
            "fichier": p.name,
            "emplacement": str(p.relative_to(TABLES)),
            "taille_Mo": round(p.stat().st_size / 1024**2, 2),
        }
        for p in fichiers
    ],
    columns=["fichier", "emplacement", "taille_Mo"],
)

print(f"Nombre de fichiers CSV trouvés : {len(fichiers)}")
display(inventaire)

In [ ]:
# Affiche uniquement les noms des colonnes, sans données individuelles.
for fichier in fichiers:
    print(f"\n{'=' * 60}")
    print(fichier.name)

    try:
        entete = pd.read_csv(
            fichier,
            nrows=0,
            encoding="utf-8-sig",
        )

        print(f"Nombre de colonnes : {len(entete.columns)}")
        print("Colonnes :", " | ".join(entete.columns))

        if len(entete.columns) == 1:
            print("À vérifier : le séparateur du fichier est peut-être différent.")

    except Exception as erreur:
        print(f"Lecture impossible : {type(erreur).__name__} — {erreur}")

In [ ]:
# Lecture des identifiants uniquement.
# Aucune donnée individuelle n'est affichée.

sources = {
    "Diagnostic": ("All_Subjects_DXSUM_", "PTID"),
    "Demographie": ("All_Subjects_PTDEMOG_", "PTID"),
    "MMSE": ("All_Subjects_MMSE_", "PTID"),
    "IRM_catalogue": ("All_Subjects_Key_MRI_", "subject_id"),
    "PET_catalogue": ("All_Subjects_Key_PET_", "subject_id"),
    "IRM_recherche": ("idaSearch_", "Subject ID"),
}

participants = {}
bilan = []

for nom, (prefixe, colonne) in sources.items():
    correspondances = [
        p for p in fichiers if p.name.startswith(prefixe)
    ]

    if len(correspondances) != 1:
        raise ValueError(
            f"{nom} : {len(correspondances)} fichiers trouvés, "
            "un seul attendu."
        )

    df = pd.read_csv(
        correspondances[0],
        usecols=[colonne],
        dtype="string",
        encoding="utf-8-sig",
    )

    ids = df[colonne].str.strip().str.upper()
    ids = ids.replace("", pd.NA)

    participants[nom] = set(ids.dropna())

    bilan.append({
        "source": nom,
        "lignes": len(df),
        "participants_uniques": ids.nunique(),
        "identifiants_manquants": int(ids.isna().sum()),
    })

display(pd.DataFrame(bilan))

comparaisons = {
    "IRM recherche + diagnostic": [
        "IRM_recherche", "Diagnostic"
    ],
    "IRM recherche + diagnostic + MMSE": [
        "IRM_recherche", "Diagnostic", "MMSE"
    ],
    "IRM recherche + diagnostic + PET catalogue": [
        "IRM_recherche", "Diagnostic", "PET_catalogue"
    ],
}

for nom, groupes in comparaisons.items():
    communs = set.intersection(
        *(participants[groupe] for groupe in groupes)
    )
    print(f"{nom} : {len(communs)} participants")

In [ ]:
# Consulter la définition du diagnostic dans le dictionnaire ADNI.
chemins = [
    p for p in fichiers
    if p.name.upper().startswith("DATADIC_")
]

if len(chemins) != 1:
    raise ValueError("Un seul fichier DATADIC est attendu.")

dictionnaire = pd.read_csv(
    chemins[0],
    dtype="string",
    encoding="utf-8-sig",
)

masque = (
    dictionnaire["TBLNAME"].str.strip().str.upper().eq("DXSUM")
    & dictionnaire["FLDNAME"].str.strip().str.upper().eq("DIAGNOSIS")
).fillna(False)

colonnes = [
    "PHASE", "TBLNAME", "FLDNAME",
    "TEXT", "CODE", "MAPPING_NOTES"
]

resultat = dictionnaire.loc[masque, colonnes].drop_duplicates()

if resultat.empty:
    print("Aucune définition trouvée avec ces critères.")
else:
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None,
    ):
        display(resultat)

In [ ]:
chemins = [
    p for p in fichiers
    if p.name.startswith("All_Subjects_DXSUM_")
]

if len(chemins) != 1:
    raise ValueError("Un seul fichier DXSUM est attendu.")

dx = pd.read_csv(
    chemins[0],
    dtype="string",
    encoding="utf-8-sig",
)

# Nettoyage en mémoire uniquement : le CSV original reste intact.
for colonne in ["PHASE", "PTID", "DIAGNOSIS"]:
    dx[colonne] = dx[colonne].str.strip().replace("", pd.NA)

print("Nombre de lignes par phase et code diagnostique :")

repartition = pd.crosstab(
    dx["PHASE"].fillna("PHASE_MANQUANTE"),
    dx["DIAGNOSIS"].fillna("DIAGNOSTIC_MANQUANT"),
)

display(repartition)

# Examiner aussi les définitions disponibles pour les autres
# champs diagnostiques, sans supposer leur signification.
champs = ["DIAGNOSIS", "DXNORM", "DXMCI", "DXAD", "DXDDUE"]

masque = (
    dictionnaire["TBLNAME"].str.strip().str.upper().eq("DXSUM")
    & dictionnaire["FLDNAME"].str.strip().str.upper().isin(champs)
).fillna(False)

definitions = dictionnaire.loc[
    masque,
    ["PHASE", "FLDNAME", "TEXT", "CODE", "MAPPING_NOTES"]
].drop_duplicates()

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
):
    display(definitions)

In [ ]:
# Audit du suivi : aucune modification du fichier original.
suivi = dx.copy()

suivi["date"] = pd.to_datetime(
    suivi["EXAMDATE"],
    format="%Y-%m-%d",
    errors="coerce",
)

print(
    "Lignes avec date absente ou non reconnue :",
    int(suivi["date"].isna().sum()),
)

# Pour cet inventaire, retenir les codes diagnostiques 1, 2 et 3.
eligibles = suivi.loc[
    suivi["DIAGNOSIS"].isin(["1", "2", "3"])
    & suivi["PTID"].notna()
    & suivi["date"].notna()
].copy()

# Repérer les diagnostics contradictoires pour un même jour.
par_jour = eligibles.groupby(["PTID", "date"])["DIAGNOSIS"].nunique()
print(
    "Couples participant/date avec plusieurs codes différents :",
    int((par_jour > 1).sum()),
)

# Compter les dates distinctes, pas simplement les lignes.
resume_suivi = eligibles.groupby("PTID").agg(
    nombre_dates=("date", "nunique"),
    premiere_date=("date", "min"),
    derniere_date=("date", "max"),
)

resume_suivi["duree_jours"] = (
    resume_suivi["derniere_date"] - resume_suivi["premiere_date"]
).dt.days

plusieurs_dates = set(
    resume_suivi.index[resume_suivi["nombre_dates"] >= 2]
)

avec_irm = plusieurs_dates & participants["IRM_recherche"]
avec_irm_pet = avec_irm & participants["PET_catalogue"]

print("\nParticipants avec au moins deux dates :", len(plusieurs_dates))
print("Parmi eux, présents dans la recherche IRM :", len(avec_irm))
print("Parmi eux, présents aussi dans le catalogue PET :", len(avec_irm_pet))

In [ ]:
# Vérifier les dates sans afficher de données individuelles.
dates_brutes = dx["EXAMDATE"].str.strip().replace("", pd.NA)

dates_lues = pd.to_datetime(
    dates_brutes,
    format="%Y-%m-%d",
    errors="coerce",
)

audit_dates = pd.DataFrame({
    "phase": dx["PHASE"],
    "date_absente": dates_brutes.isna(),
    "date_non_reconnue": dates_brutes.notna() & dates_lues.isna(),
})

print("Bilan des dates par phase :")
display(
    audit_dates.groupby("phase")[
        ["date_absente", "date_non_reconnue"]
    ].sum()
)

# Afficher uniquement les formes des dates non reconnues :
# chaque chiffre est remplacé par N.
non_reconnues = dates_brutes[
    dates_brutes.notna() & dates_lues.isna()
]

print("\nFormats non reconnus :")
display(
    non_reconnues.str.replace(r"\d", "N", regex=True)
    .value_counts()
    .rename_axis("format")
    .reset_index(name="nombre")
)

# Résumé des conflits, sans identifiants ni dates.
cles_conflit = par_jour[par_jour > 1].index

lignes_conflit = eligibles.loc[
    pd.MultiIndex.from_frame(
        eligibles[["PTID", "date"]]
    ).isin(cles_conflit)
]

print("\nLignes impliquées dans les conflits, par phase et code :")
display(
    lignes_conflit.groupby(["PHASE", "DIAGNOSIS"])
    .size()
    .reset_index(name="nombre_lignes")
)

In [ ]:
# 1. Recalculer le suivi diagnostique sans les visites contradictoires.
index_visites = pd.MultiIndex.from_frame(
    eligibles[["PTID", "date"]]
)

dx_provisoire = eligibles.loc[
    ~index_visites.isin(cles_conflit)
].copy()

nb_dates_dx = dx_provisoire.groupby("PTID")["date"].nunique()
ids_suivi_dx = set(nb_dates_dx.index[nb_dates_dx >= 2])

# 2. Lire l'export de la recherche IRM.
exports = [
    p for p in fichiers if p.name.startswith("idaSearch_")
]

if len(exports) != 1:
    raise ValueError("Un seul export idaSearch est attendu.")

irm = pd.read_csv(
    exports[0],
    dtype="string",
    encoding="utf-8-sig",
)

irm["PTID"] = (
    irm["Subject ID"].str.strip().str.upper().replace("", pd.NA)
)

# L'export IDA utilise mois/jour/année.
irm["date"] = pd.to_datetime(
    irm["Study Date"].str.strip(),
    format="%m/%d/%Y",
    errors="coerce",
)

print(
    "Lignes IRM avec date absente ou non reconnue :",
    int(irm["date"].isna().sum()),
)

# 3. Compter les dates distinctes par participant.
irm_valides = irm.dropna(subset=["PTID", "date"])

nb_dates_irm = irm_valides.groupby("PTID")["date"].nunique()
ids_suivi_irm = set(nb_dates_irm.index[nb_dates_irm >= 2])

candidats_irm = ids_suivi_irm & ids_suivi_dx
candidats_irm_pet = candidats_irm & participants["PET_catalogue"]

bilan_longitudinal = pd.DataFrame([
    {
        "critere": "Au moins 2 dates de diagnostic sans conflit",
        "participants": len(ids_suivi_dx),
    },
    {
        "critere": "Au moins 2 dates d'IRM",
        "participants": len(ids_suivi_irm),
    },
    {
        "critere": "Au moins 2 dates d'IRM et 2 dates de diagnostic",
        "participants": len(candidats_irm),
    },
    {
        "critere": "Critère précédent + présence au catalogue PET",
        "participants": len(candidats_irm_pet),
    },
])

display(bilan_longitudinal)

In [ ]:
# Une ligne par participant et date d'IRM.
visites_irm = (
    irm_valides.loc[
        irm_valides["PTID"].isin(candidats_irm),
        ["PTID", "date"]
    ]
    .drop_duplicates()
    .rename(columns={"date": "date_irm"})
)

# Une ligne par participant et date de diagnostic sans conflit.
visites_dx = (
    dx_provisoire.loc[
        dx_provisoire["PTID"].isin(candidats_irm),
        ["PTID", "date"]
    ]
    .drop_duplicates()
    .rename(columns={"date": "date_dx"})
)

# Audit exploratoire : chercher le diagnostic le plus proche,
# avant OU après l'IRM.
appariements = pd.merge_asof(
    visites_irm.sort_values("date_irm"),
    visites_dx.sort_values("date_dx"),
    by="PTID",
    left_on="date_irm",
    right_on="date_dx",
    direction="nearest",
)

appariements["ecart_jours"] = (
    appariements["date_irm"] - appariements["date_dx"]
).dt.days.abs()

resultats = []

for seuil in [0, 30, 90, 180]:
    proches = appariements.loc[
        appariements["ecart_jours"] <= seuil
    ]

    comptes = proches.groupby("PTID").agg(
        dates_irm=("date_irm", "nunique"),
        dates_dx=("date_dx", "nunique"),
    )

    # Exiger deux dates distinctes de chaque côté :
    # un seul diagnostic ne doit pas suffire pour deux IRM.
    ids = set(comptes.index[
        (comptes["dates_irm"] >= 2)
        & (comptes["dates_dx"] >= 2)
    ])

    resultats.append({
        "ecart_maximal_jours": seuil,
        "participants_avec_2_dates_appariees": len(ids),
        "dont_presents_catalogue_PET": len(
            ids & participants["PET_catalogue"]
        ),
    })

display(pd.DataFrame(resultats))

In [ ]:
chemins_pet = [
    p for p in fichiers
    if p.name.startswith("All_Subjects_Key_PET_")
]

if len(chemins_pet) != 1:
    raise ValueError("Un seul fichier Key_PET est attendu.")

pet = pd.read_csv(
    chemins_pet[0],
    dtype="string",
    encoding="utf-8-sig",
)

pet["PTID"] = (
    pet["subject_id"]
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

# Les 1 430 candidats avant application d'un seuil temporel.
pet_candidats = pet.loc[
    pet["PTID"].isin(candidats_irm_pet)
].copy()

print("Types de PET disponibles chez les participants candidats :")

for colonne in ["radiopharmaceutical", "amyloid_pet", "tau_pet"]:
    valeurs = (
        pet_candidats[colonne]
        .str.strip()
        .replace("", pd.NA)
        .fillna("NON_RENSEIGNE")
    )

    bilan = (
        pet_candidats.assign(valeur=valeurs)
        .groupby("valeur")
        .agg(
            lignes=("PTID", "size"),
            participants=("PTID", "nunique"),
        )
        .sort_values("participants", ascending=False)
        .reset_index()
    )

    print(f"\n{colonne}")
    display(bilan)

In [ ]:
# Sélection provisoire : un seul traceur.
fdg = pet_candidats.loc[
    pet_candidats["radiopharmaceutical"].str.strip().eq("18F-FDG")
].copy()

dates_brutes = fdg["image_date"].str.strip().replace("", pd.NA)

fdg["date_pet"] = pd.to_datetime(
    dates_brutes,
    format="%Y-%m-%d",
    errors="coerce",
)

absentes = dates_brutes.isna()
non_reconnues = dates_brutes.notna() & fdg["date_pet"].isna()

print("Dates PET-FDG absentes :", int(absentes.sum()))
print("Dates PET-FDG non reconnues :", int(non_reconnues.sum()))

if non_reconnues.any():
    print("Formats à vérifier :")
    display(
        dates_brutes[non_reconnues]
        .str.replace(r"\d", "N", regex=True)
        .value_counts()
    )
else:
    visites_fdg = (
        fdg.dropna(subset=["PTID", "date_pet"])
        [["PTID", "date_pet"]]
        .drop_duplicates()
    )

    # Audit : IRM la plus proche de chaque date PET,
    # avant ou après le PET.
    liens_pet_irm = pd.merge_asof(
        visites_fdg.sort_values("date_pet"),
        visites_irm.sort_values("date_irm"),
        by="PTID",
        left_on="date_pet",
        right_on="date_irm",
        direction="nearest",
    )

    liens_pet_irm["ecart_jours"] = (
        liens_pet_irm["date_pet"] - liens_pet_irm["date_irm"]
    ).dt.days.abs()

    bilan = []

    for seuil in [30, 90, 180]:
        proches = liens_pet_irm.loc[
            liens_pet_irm["ecart_jours"] <= seuil
        ]

        comptes = proches.groupby("PTID").agg(
            dates_pet=("date_pet", "nunique"),
            dates_irm=("date_irm", "nunique"),
        )

        bilan.append({
            "ecart_maximal_jours": seuil,
            "participants_avec_au_moins_1_paire": len(comptes),
            "participants_avec_2_dates_de_chaque_modalite": int(
                (
                    (comptes["dates_pet"] >= 2)
                    & (comptes["dates_irm"] >= 2)
                ).sum()
            ),
        })

    display(pd.DataFrame(bilan))

In [ ]:
import json

SEUIL_PROVISOIRE = 90

paires_provisoires = liens_pet_irm.loc[
    liens_pet_irm["ecart_jours"] <= SEUIL_PROVISOIRE
].copy()

comptes = paires_provisoires.groupby("PTID").agg(
    nombre_dates_pet=("date_pet", "nunique"),
    nombre_dates_irm=("date_irm", "nunique"),
)

candidats = comptes.loc[
    (comptes["nombre_dates_pet"] >= 2)
    & (comptes["nombre_dates_irm"] >= 2)
].reset_index()

# Un dossier distinct pour chaque exécution :
# les résultats précédents ne sont pas écrasés.
execution = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")

DOSSIER_AUDIT = (
    ROOT / "data" / "05_Manifestes" / f"audit_{execution}"
)
DOSSIER_AUDIT.mkdir(parents=True, exist_ok=False)

candidats.to_csv(
    DOSSIER_AUDIT / "candidats_provisoires.csv",
    index=False,
)

paires_provisoires.loc[
    paires_provisoires["PTID"].isin(candidats["PTID"])
].to_csv(
    DOSSIER_AUDIT / "rapprochements_irm_pet_provisoires.csv",
    index=False,
)

configuration = {
    "statut": "exploratoire_non_valide",
    "traceur": "18F-FDG",
    "ecart_maximal_jours": SEUIL_PROVISOIRE,
    "minimum_dates_distinctes_par_modalite": 2,
    "appariement": "IRM la plus proche de chaque PET, avant ou apres",
    "diagnostics_des_visites_appariees_verifies": False,
    "nombre_candidats": len(candidats),
    "fichiers_sources": [p.name for p in fichiers],
}

(DOSSIER_AUDIT / "configuration_audit.json").write_text(
    json.dumps(configuration, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"{len(candidats)} candidats provisoires enregistrés.")
print("Dossier :", DOSSIER_AUDIT)